# CrewAI Trip Planner
This notebook condenses the multi-file project into a single, straightforward procedural flow.

In [6]:
import os
!pip install crewai unstructured pyowm python-dotenv langchain-openai

In [11]:
import os
import json
import requests
from textwrap import dedent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from crewai import Agent, Task, Crew
from dotenv import load_dotenv

load_dotenv()

False

In [12]:
# --- 1. DEFINE TOOLS ---
from google.colab import userdata
SERPER_API_KEY = userdata.get('SERPER_API_KEY')

@tool("Search the internet")
def search_internet(query: str) -> str:
    """Useful to search the internet about a given topic and return relevant results."""
    top_result_to_return = 4
    url = "https://google.serper.dev/search"
    payload = json.dumps({"q": query})
    headers = {
        'X-API-KEY': SERPER_API_KEY,
        'content-type': 'application/json'
    }
    response = requests.request("POST", url, headers=headers, data=payload)

    if 'organic' not in response.json():
        return "Error: Could not retrieve search results. Please check your Serper API key."

    results = response.json()['organic']
    string = []
    for result in results[:top_result_to_return]:
        try:
            string.append('\n'.join([
                f"Title: {result['title']}", f"Link: {result['link']}",
                f"Snippet: {result['snippet']}", "\n-----------------"
            ]))
        except KeyError:
            continue
    return '\n'.join(string)

@tool("Make a calculation")
def calculate(operation: str):
    """This tool allows you to perform mathematical operations like
    addition, subtraction, multiplication, and division.
    It takes a mathematical expression as input, such as 150+25 or 300/5*2."""
    try:
        return eval(operation)
    except SyntaxError:
        return "Error: Invalid syntax in mathematical expression"

In [13]:
# --- 2. INITIALIZE LLM ---
from google.colab import userdata

# Make sure you have 'OPENAI_API_KEY' set in your Colab Secrets
llm = ChatOpenAI(
    api_key=userdata.get('OPENAI_API_KEY'),
    model="gpt-5.4-mini"
)

In [15]:
# --- 3. DEFINE AGENTS ---
# Using the updated ChatOpenAI instance 'llm'
# We pass tools directly; if validation fails, ensure crewai and langchain are compatible versions.
expert_travel_agent = Agent(
    role="Expert Travel Agent",
    backstory="I'm a seasoned expert in travel planning and logistics. With decades of experience, I'm here to help plan your perfect trip.",
    goal="Plan a 7-day trip with day-by-day details, including a budget, packing tips, and safety recommendations.",
    tools=[search_internet, calculate],
    verbose=True,
    llm=llm,
    allow_delegation=False
)

city_selection_expert = Agent(
    role="City Selection Expert",
    backstory="A pro at analyzing travel data to pick the best destinations based on a variety of factors.",
    goal="Find the top cities to visit, considering weather, seasons, budget, and what the traveler enjoys.",
    tools=[search_internet],
    verbose=True,
    llm=llm,
    allow_delegation=False
)

local_tour_guide = Agent(
    role="Local Tour Guide",
    backstory="A knowledgeable local guide who knows everything about the city's attractions, culture, and hidden gems.",
    goal="Give the best insights and recommendations about the selected city for an unforgettable experience.",
    tools=[search_internet],
    verbose=True,
    llm=llm,
    allow_delegation=False
)

ValidationError: 4 validation errors for Agent
tools.0
  Input should be a valid dictionary or instance of BaseTool [type=model_type, input_value=StructuredTool(name='Sear...rnet at 0x7c261b606340>), input_type=StructuredTool]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type
tools.1
  Input should be a valid dictionary or instance of BaseTool [type=model_type, input_value=StructuredTool(name='Make...late at 0x7c261b6f5800>), input_type=StructuredTool]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type
llm.str
  Input should be a valid string [type=string_type, input_value=ChatOpenAI(metadata={'lc_...eam_chunk_timeout=120.0), input_type=ChatOpenAI]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type
llm.BaseLLM
  Input should be a valid dictionary or instance of BaseLLM [type=model_type, input_value=ChatOpenAI(metadata={'lc_...eam_chunk_timeout=120.0), input_type=ChatOpenAI]
    For further informat

In [ ]:
# --- 4. GATHER USER INPUT ---
print("## Welcome to Trip Planner Crew")
print('-------------------------------')

origin = input("From where will you be traveling from?\n")
cities = input("What are the cities options you are interested in visiting?\n")
date_range = input("What is the date range you are interested in traveling?\n")
interests = input("What are some of your high-level interests and hobbies?\n")

## Welcome to Trip Planner Crew
-------------------------------
From where will you be traveling from?
Bangalore
What are the cities options you are interested in visiting?
Oslo, Vancouver, Tokyo
What is the date range you are interested in traveling?
First week of July
What are some of your high-level interests and hobbies?
Mountains, nature, skyline


In [ ]:
# --- 5. DEFINE TASKS ---
destination_selection_task = Task(
    description=dedent(f"""**Task**: Choose the Best Travel Destination
        **Objective**: Evaluate multiple cities and select the most suitable destination
        based on weather, events, and travel costs. The decision should be backed by real data.

        **Key Considerations**:
           - Compare weather forecasts, seasonal festivals, and cultural events.
           - Analyze estimated travel costs, including flights and accommodations.
           - Ensure the chosen city aligns with the traveler's preferences and interests.
           - Provide a detailed report on the selected destination, including key attractions.

        **Travel Details**:
           - Starting Location: {origin}
           - Destination Options: {cities}
           - Traveler Interests: {interests}
           - Travel Dates: {date_range}"""),
    agent=city_selection_expert
)

city_guide_task = Task(
    description=dedent(f""" **Task**: Compile a Comprehensive City Guide
        **Objective**: Gather essential details about {cities}, including top attractions, local customs,
        seasonal events, and must-visit spots. The guide should be practical, informative, and engaging.
        **Key Insights to Include**:
                - Major landmarks, hidden gems, and unique cultural experiences.
                - Local customs, traditions, and etiquette.
                - Special events or festivals occurring during the travel dates.
                - Weather forecast and high-level cost estimates for accommodations and activities.
            **Travel Details**:
                - City: {cities}
                - Traveler Interests: {interests}
                - Travel Dates: {date_range}"""),
    agent=local_tour_guide
)

itinerary_task = Task(
    description=dedent(f"""**Task**: Develop a 7-Day Travel Itinerary
        **Description**: Expand the city guide into a full 7-day travel itinerary
        with detailed per-day plans, including weather forecasts, places to eat, packing suggestions,
        and a budget breakdown. You MUST suggest actual places to visit, actual hotels to stay,
        and actual restaurants to go to. This itinerary should cover all aspects of the trip,
        from arrival to departure, integrating the city guide information with practical travel logistics.
        **Parameters**:
        - City: {cities}
        - Trip Date: {date_range}
        - Traveler Interests: {interests}"""),
    agent=expert_travel_agent
)

In [ ]:
# --- 6. ASSEMBLE AND RUN THE CREW ---
travel_crew = Crew(
    agents=[expert_travel_agent, city_selection_expert, local_tour_guide],
    tasks=[itinerary_task, destination_selection_task, city_guide_task],
    verbose=True
)

result = travel_crew.kickoff()

print("\n\n########################")
print("## Here is your Trip Plan")
print("########################\n")
print(result)

Working Agent: Expert Travel Agent
Starting Task: **Task**: Develop a 7-Day Travel Itinerary
        **Description**: Expand the city guide into a full 7-day travel itinerary
        with detailed per-day plans, including weather forecasts, places to eat, packing suggestions,
        and a budget breakdown. You MUST suggest actual places to visit, actual hotels to stay,
        and actual restaurants to go to. This itinerary should cover all aspects of the trip,
        from arrival to departure, integrating the city guide information with practical travel logistics.
        **Parameters**:
        - City: Oslo, Vancouver, Tokyo
        - Trip Date: First week of July
        - Traveler Interests: Mountains, nature, skyline


> Entering new CrewAgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Search the internet
Action Input: weather forecast for Oslo, Vancouver, Tokyo in the first week of JulyTitle: How bad is the weather in July really? : r/JapanTravelTips - Reddi